# YOLO models

In [8]:
import glob
import random
import os
from pathlib import Path
from PIL import Image

# Repo root discovery: works whether the kernel cwd is the repo root or examples/
ROOT = Path.cwd()
while not (ROOT / "utils" / "models.py").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

import os
from PIL import Image

from utils.models import Marin2025Detector, Stark2023Detector, Bjerge2023Detector, FlatBugDetector, Detection
from utils.viz_utils import plot_results_masks, plot_masks_zoomed, plot_results, plot_crop_zoomed

In [26]:
detector = Marin2025Detector(model_name=f"{ROOT}/models-cache/marin2025.pt")
# detector = Stark2023Detector(model_name=f"{ROOT}/models-cache/yolov5_s_best_stark2023.pt")
# detector = Bjerge2023Detector(model_name=f"{ROOT}/models-cache/insect1201-bestF1-1280v5m6_bjerge2023.pt")
# detector = FlatBugDetector(model_name=f"{ROOT}/models-cache/flat_bug_M.pt")

print(f"supports_mask = {detector.supports_mask}")

supports_mask = False


In [ ]:
from utils.inference import has_binary_classifier, load_binary_classifier

import glob
import random

image_path = random.choice(sorted(glob.glob(os.path.join(str(ROOT / "example_dataset"), "**", "*.jpg"), recursive=True)))

image = Image.open(image_path).convert("RGB")

print(f"Image width: {image.width}, height: {image.height}")

conf_threshold = 0.1
final_det_threshold = 0.1

# Optional: binary classifier (insect vs background). Pick the combination you want
USE_BINARY_CLASSIFIER = False
DISCRIMINATOR_MODE = "mlp"  # "mlp" or "svm"
DISCRIMINATOR_NAME = "dinov3"  # "dinov3" or "bioclip2_5"

classifier = None
if USE_BINARY_CLASSIFIER and has_binary_classifier(DISCRIMINATOR_MODE, DISCRIMINATOR_NAME):
    classifier = load_binary_classifier(DISCRIMINATOR_MODE, DISCRIMINATOR_NAME)
    print(f"Binary classifier: {DISCRIMINATOR_MODE} / {DISCRIMINATOR_NAME}")
elif USE_BINARY_CLASSIFIER:
    print(f"Binary classifier weights not found: {DISCRIMINATOR_MODE} / {DISCRIMINATOR_NAME}. Train them first")

# How to combine detection score with the binary classifier: "joint" (product) or "mean" (average)
BINARY_SCORE_MODE = "joint"  # try "mean"
FIXRES = 0.3  # 30% margin around the box before classification

results: list[Detection] = detector.predict(
    image=image,
    conf_threshold=conf_threshold,
    slice_size=None,
    iou_threshold=0.1,
    overlap=0.2,
)

# Binary classifier: applies only if weights are present
if classifier is not None:
    for d in results:
        x1, y1, x2, y2 = [int(v) for v in d.box]
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue
        # FIXRES: enlarge the box by a 30% margin before classification, clamped to the image
        dx, dy = (x2 - x1) * FIXRES, (y2 - y1) * FIXRES
        crop = image.crop((max(0, x1 - dx), max(0, y1 - dy),
                           min(image.width, x2 + dx), min(image.height, y2 + dy)))
        insect_proba = classifier.predict(crop).score
        if BINARY_SCORE_MODE == "joint":
            d.score = d.score * insect_proba
        elif BINARY_SCORE_MODE == "mean":
            d.score = (d.score + insect_proba) / 2

print("Final detections before thresholding:")
for d in results:
    print(f"  - {d.label}: {d.score:.2f}")

results = [d for d in results if round(d.score, 2) >= final_det_threshold]

plot_results(image_path, results)
plot_crop_zoomed(image_path, results)